# Group A2: Lab Test – Seq2Seq Preprocessing, Encoder-Decoder Model, and Decoding Inference

### Task Objectives:
1. **Preprocessing**: Construct target vocabulary containing `<SOS>` (Start of Sentence) and `<EOS>` (End of Sentence) tokens.
2. **Model Architecture**: Build `EncoderRNN` and `DecoderRNN` modules.
3. **Inference**: Implement greedy step-by-step decoding in `translate_and_evaluate()`.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

# Toy Sequence Dataset (Reversing words)
pairs = [
    ("one two three", "three two one"),
    ("cat sat mat", "mat sat cat"),
    ("red blue green", "green blue red")
]

# =============================================================================
# PART 1: PREPROCESSING (Student TODO)
# =============================================================================
def build_seq2seq_vocab(pairs):
    """
    TODO: Build unified vocab dict with '<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3.
    """
    vocab = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
    # YOUR CODE HERE
    for src, tgt in pairs:
        for word in src.split() + tgt.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

def encode_sequence(seq, vocab, add_sos_eos=False):
    """
    TODO: Encode token string into list of IDs. Append <SOS>/<EOS> if add_sos_eos=True.
    """
    # YOUR CODE HERE
    tokens = seq.split() if isinstance(seq, str) else seq
    ids = [vocab.get(tok, vocab.get("<UNK>", 1)) for tok in tokens]
    if add_sos_eos:
        ids = [vocab["<SOS>"]] + ids + [vocab["<EOS>"]]
    return ids

# =============================================================================
# PART 2: MODEL ARCHITECTURE (Student TODO)
# =============================================================================
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        # TODO: Define Embedding and RNN/GRU
        # YOUR CODE HERE
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        # TODO: Return outputs and final hidden state
        # YOUR CODE HERE
        outputs, hidden = self.rnn(self.embedding(x))
        return outputs, hidden

class DecoderRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        # TODO: Define Embedding, RNN/GRU, and Linear layer to vocab_size
        # YOUR CODE HERE
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        # TODO: Execute 1 time-step forward pass. Return logits (Batch, 1, vocab_size), hidden
        # YOUR CODE HERE
        output, hidden = self.rnn(self.embedding(x), hidden)
        logits = self.fc(output)
        return logits, hidden

# =============================================================================
# PART 3: INFERENCE DECODING LOOP (Student TODO)
# =============================================================================
def translate_and_evaluate(encoder, decoder, input_seq, vocab, inv_vocab, max_steps=5):
    """
    TODO: Perform greedy autoregressive decoding:
    1. Pass input_seq through encoder to obtain hidden context.
    2. Start decoder input with <SOS> token ID.
    3. Iteratively generate tokens until <EOS> or max_steps reached.
    Return generated string prediction.
    """
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        # YOUR CODE HERE
        src_ids = torch.tensor([encode_sequence(input_seq, vocab)], dtype=torch.long).to(device)
        _, dec_hidden = encoder(src_ids)
        dec_in = torch.tensor([[vocab["<SOS>"]]], dtype=torch.long).to(device)
        predicted_tokens = []
        for _ in range(max_steps):
            dec_out, dec_hidden = decoder(dec_in, dec_hidden)
            pred_id = dec_out.argmax(dim=-1).item()
            if pred_id == vocab["<EOS>"]:
                break
            predicted_tokens.append(inv_vocab.get(pred_id, "<UNK>"))
            dec_in = torch.tensor([[pred_id]], dtype=torch.long).to(device)
        return " ".join(predicted_tokens)

# =============================================================================
# PART 4: AUTOMATED VERIFICATION SUITE (Do Not Modify)
# =============================================================================
def run_test():
    vocab = build_seq2seq_vocab(pairs)
    inv_vocab = {v: k for k, v in vocab.items()}
    
    encoder = EncoderRNN(len(vocab)).to(device)
    decoder = DecoderRNN(len(vocab)).to(device)
    
    enc_opt = optim.Adam(encoder.parameters(), lr=0.01)
    dec_opt = optim.Adam(decoder.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    # Overfit Training Loop
    for epoch in range(60):
        for src, tgt in pairs:
            src_ids = torch.tensor([encode_sequence(src, vocab)], dtype=torch.long).to(device)
            tgt_ids = encode_sequence(tgt, vocab, add_sos_eos=True)
            
            enc_opt.zero_grad()
            dec_opt.zero_grad()
            
            _, enc_hidden = encoder(src_ids)
            dec_hidden = enc_hidden
            
            loss = 0
            for t in range(len(tgt_ids) - 1):
                dec_in = torch.tensor([[tgt_ids[t]]], dtype=torch.long).to(device)
                dec_out, dec_hidden = decoder(dec_in, dec_hidden)
                target = torch.tensor([tgt_ids[t+1]], dtype=torch.long).to(device)
                loss += criterion(dec_out.squeeze(1), target)
                
            loss.backward()
            enc_opt.step()
            dec_opt.step()
            
    # Test Prediction
    test_src = "cat sat mat"
    predicted_tgt = translate_and_evaluate(encoder, decoder, test_src, vocab, inv_vocab)
    print(f"Input: '{test_src}' | Predicted Output: '{predicted_tgt}'")
    
    assert "mat" in predicted_tgt and "cat" in predicted_tgt, "Inference Failed: Sequence generated improperly."
    print("Lab task passed!")

run_test()

Input: 'cat sat mat' | Predicted Output: 'mat sat cat'
Lab task passed!
